# Lecture: Training a Small Language Model (size, speed, projections)

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Explain tokens, training loss, and validation loss in one sentence each — and why a growing train/val gap means memorization, not learning (§4).
- Convert any loss into perplexity with `perplexity = exp(loss)` and read it as the number of characters the model hesitates between (§5).
- Explain why tokens/sec falls as models grow, and how one measured config projects 100M / 300M / 1B memory and speed (§6–§7).
- Read the system map (§8) and narrate the lab loop: config → train → loss dashboard → projection.


## The big idea

A language model is a student taking a fill-in-the-blank quiz millions of times: it sees “to be or not to ” and guesses the next letter. Training means checking each guess with a wrongness score (the **loss**) and nudging the student's knobs (the **parameters**) toward better guesses. A bigger student memorizes more patterns, so its guesses improve — but every quiz question needs more multiplications, so training speed (**tokens per second**) drops. The lab trains three tiny students (XS / S / M, 0.1M → 1.2M knobs) for real, then uses their numbers as an anchor: memory is just bytes-per-knob times knob count, and speed falls roughly as 1 / knob count — so we can project what a 100M, 300M, or 1B model would cost without ever training one.


In [ ]:
import numpy as np  # bring in numpy for the small config arrays
import matplotlib.pyplot as plt  # bring in pyplot for the two-panel dashboard sketch
np.random.seed(13)  # fix the seed so the lecture is identical every run
names = ['XS', 'S', 'M']  # short labels for the three proxy configs in the lab
params = np.array([109824, 460416, 1204736])  # knob counts per config (exact lab sizes)
val = np.array([2.7944, 2.6177, 2.6060])  # measured val loss per config (lower is better)
toks = np.array([163000, 62000, 30000])  # measured tokens/sec per config (higher is faster)
print('params:', params.tolist())  # show exact model sizes plotted below
print('val loss:', val.tolist())  # show loss falls as the model grows
print('tok/s:', toks.tolist())  # show speed falls as the model grows
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))  # make two side-by-side panels
ax[0].plot(names, val, marker='o')  # left: bigger model, lower loss (quality win)
ax[0].set_title('Bigger model -> lower loss')  # title states the quality win
ax[0].set_ylabel('val loss (lower is better)')  # label defines the loss direction
ax[0].set_xlabel('model size')  # label names the x axis simply
ax[0].grid(True, alpha=0.3)  # add faint grid to read values easily
ax[1].bar(names, toks, color='tab:orange')  # right: bigger model, slower speed (price)
ax[1].set_title('Bigger model -> slower tok/s')  # title states the speed cost
ax[1].set_ylabel('tokens/sec (higher is faster)')  # label defines the speed direction
ax[1].set_xlabel('model size')  # label names the x axis simply
fig.tight_layout()  # stop titles and labels from overlapping
plt.show()  # display the dashboard sketch inside the notebook
print('Lesson: size buys quality but costs speed.')  # one-sentence takeaway
print('RECALIBRATE: constants above are the author CPU run (SEED=13); paste your results/configs.csv + projections.csv row to replace them - ordering/shape is the claim, digits are not')


## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 scores the guesses, §5 rescales the score, §6–§7 turn measurements into predictions.

| Term | One-liner | Section |
|---|---|---|
| Token | One chunk the model reads at a time (here: a single character like `t` or ` `) | §4 |
| Loss (cross-entropy) | One number for “how bad were the guesses”; lower is better | §4 |
| Train loss | Loss on the text the model studies (can be gamed by memorizing) | §4 |
| Val loss / overfit gap | Loss on fresh held-out text; gap = val − train (memorization alarm) | §4 |
| Perplexity | `exp(loss)` — roughly “how many letters the model hesitates between” | §5 |
| Throughput (tok/s) | Tokens processed per second; falls ~1/params as models grow | §6 |
| Projection | Stretch small-model measurements to 100M/300M/1B with bytes-per-knob math | §7 |


### §4 Tokens, loss, and validation — scoring the guesses honestly

**In one sentence: a token is one character the model reads, the loss is one number scoring how bad its next-character guesses were, and validation loss re-scores on fresh text the model never studied so memorization cannot hide.**

A **token** here is a single character (letters, spaces, punctuation — about 65 distinct ones). The **loss** (cross-entropy) averages “−log(probability given to the true next character)” over thousands of positions: confident-and-right is near 0, confident-and-wrong is huge. **Train loss** almost always falls — but a giant model can just memorize the answers. So we hold out the last 10% of the book and score **val loss** there; the **overfit gap** (val − train) is the memorization alarm. Watch it (SKETCH — not lab data: the lab trains XS/S/M for only 150/120/80 iters and records one point each, so you will see about 0 val gap; overfitting needs longer runs — try 5x iters as a what-if): both curves fall together, then val bends back up while train keeps falling — the final gap is 0.57 (train 0.48, val 1.05), and val is best (0.99) at step 100.


In [ ]:
import numpy as np  # reuse numpy for the synthetic loss curves
import matplotlib.pyplot as plt  # reuse pyplot for the train/val plot
np.random.seed(13)  # fix the seed so the curves match every run
steps = np.arange(0, 121, 10)  # training checkpoints: step 0, 10, ..., 120
train = 4.17 * np.exp(-steps / 35.0) + 0.35  # SKETCH curve (init 4.17 = ln(65) uniform baseline): falls fast, then flattens (memorizes)
gap = 0.02 + (steps / 120.0) ** 2 * 0.55  # memorization gap: tiny early, 0.57 by step 120
val = train + gap  # val curve: same learning plus the memorization penalty
print('final train loss:', round(float(train[-1]), 2))  # print 0.48: studying looks great
print('final val loss:', round(float(val[-1]), 2))  # print 1.05: fresh text disagrees
print('overfit gap:', round(float(gap[-1]), 2))  # print 0.57: the memorization alarm
print('best val:', round(float(val.min()), 2), 'at step', int(steps[val.argmin()]))  # print 0.99 at step 100: stop here
fig, ax = plt.subplots(figsize=(6, 4))  # make one figure for the two curves
ax.plot(steps, train, marker='o', ms=4, label='train loss')  # draw the studying curve (keeps falling)
ax.plot(steps, val, marker='s', ms=4, label='val loss')  # draw the honesty curve (bends up)
ax.fill_between(steps, train, val, alpha=0.2, label='overfit gap')  # shade the memorization zone
ax.set_xlabel('training steps')  # label the x-axis (study time)
ax.set_ylabel('loss (lower is better)')  # label the wrongness direction
ax.set_title('Train falls forever; val bends up (gap 0.57)')  # title the moral
ax.legend()  # show which curve is which
plt.tight_layout()  # snug the layout so labels are not clipped
plt.show()  # render the figure

### §5 Perplexity — exp(loss), the hesitation count

**In one sentence: perplexity is `exp(loss)` — roughly how many characters the model hesitates between when guessing — so loss 4.11 means ~61 live options and loss 2.61 means ~14.**

The loss is a logarithm, which squishes big differences into small decimals — hard to feel. Exponentiating undoes the squish. A model guessing uniformly at random over 61 characters has loss ln(61) ≈ 4.11 and perplexity 61 (maximum confusion). The lab's XS proxy reaches loss 2.79 → perplexity 16.35 (hesitates between ~16 letters), and S reaches 2.62 → 13.7. Same training, friendlier units. Watch it: the three bars below print exactly those numbers.


In [ ]:
import numpy as np  # reuse numpy for the bar positions
import math  # bring in math for exp(): perplexity = e^loss
import matplotlib.pyplot as plt  # reuse pyplot for the perplexity bars
np.random.seed(13)  # fix the seed for reproducibility
losses = [4.11, 2.7944, 2.6177, 2.6060]  # random-guess ln(61), then XS, S, M val losses
names = ['random', 'XS', 'S', 'M']  # labels for the four checkpoints
ppls = [round(math.exp(x), 2) for x in losses]  # undo the log: e^loss = hesitation count
print('losses:', losses)  # print raw wrongness going down
print('perplexities:', ppls)  # print hesitation shrinking 60.95 -> 13.54
fig, ax = plt.subplots(figsize=(6, 4))  # make one figure for the bars
ax.bar(names, ppls, color=['gray', 'tab:blue', 'tab:green', 'tab:green'])  # draw shrinking confusion
ax.set_ylabel('perplexity (fewer options = better)')  # label defines perplexity simply
ax.set_xlabel('checkpoint')  # label names the training stage
ax.set_title('Training shrinks confusion: 60.95 -> 13.54 options')  # title in printed numbers
plt.tight_layout()  # fix any label overlap before drawing
plt.show()  # render the bar chart

### §6 Throughput and scaling — why bigger means slower

**In one sentence: throughput (tokens/sec) is total tokens divided by wall seconds, and it falls as models grow because every token pays one multiply per knob — so `tok/s × params ≈ constant` within ~2x.**

Each training step runs the whole model over `batch × context` tokens and times it with a wall clock; divide and you get tok/s. Doubling the knobs roughly doubles the multiplies per token, so speed halves: XS (109,824 knobs) does 163,000 tok/s while M (1,204,736 knobs) does 30,000. Multiply each pair — 1.79e10, 2.85e10, 3.61e10 — and they agree within ~2x, good enough for back-of-envelope predictions. (The lab also sweeps batch size 8→64: speed climbs, then flattens once the CPU is saturated.) Watch it: the bars plus the near-constant product below.


In [ ]:
import numpy as np  # reuse numpy for the throughput arrays
import matplotlib.pyplot as plt  # reuse pyplot for the size-vs-speed bars
np.random.seed(13)  # fix the seed for reproducibility
names = ['XS', 'S', 'M']  # the three proxy configs again
params = np.array([109824, 460416, 1204736])  # knob counts (same as the anchor cell)
toks = np.array([163000, 62000, 30000])  # measured tok/s (same as the anchor cell)
const = toks * params  # fitted law: tok/s x params should be roughly constant
print('tok/s x params:', const.tolist())  # print 1.79e10, 2.85e10, 3.61e10: within ~2x
print('S per-token us:', round(1e6 / 62000, 1))  # print 16.1 us: one S token costs 16.1 microseconds
fig, ax = plt.subplots(figsize=(6, 4))  # make one figure for the bars
ax.bar(names, toks, color='tab:orange')  # draw speed collapsing with size
ax.set_ylabel('tokens/sec (higher is faster)')  # label defines the speed direction
ax.set_xlabel('model size')  # label names the x axis simply
ax.set_title('Bigger model -> slower tok/s (163000 -> 30000)')  # title in printed numbers
plt.tight_layout()  # fix any label overlap before drawing
plt.show()  # render the bars

### §7 Projection — stretching one measurement to 100M / 300M / 1B

**In one sentence: a projection multiplies knob counts by bytes-per-knob for memory (2 bytes inference, 14 bytes training) and divides the fitted speed constant by knob count for tok/s — no giant training run needed.**

Memory is honest multiplication: each knob stored in bf16 costs 2 bytes to *run* the model, and ~14 bytes to *train* it (weights + gradients + Adam's two optimizer states). Speed uses the §6 law with the S config as anchor (constant 2.85e10): 100M knobs → 285.5 tok/s on this CPU, 300M → 95.2, 1B → 28.5. So a 1B training run needs ~14.0 GB just for numbers (plus activations) and crawls at ~28 tok/s here — which is exactly why the lab says “rerun on the DGX Spark, tok/s jumps 50–100x”. Watch it: the extrapolation line plus the memory bars, all printed.


In [ ]:
import numpy as np  # reuse numpy for the projection arrays
import matplotlib.pyplot as plt  # reuse pyplot for line + bars
np.random.seed(13)  # fix the seed for reproducibility
rate = 62000 * 460416  # fitted speed constant from the S anchor (tok/s x params)
sizes = np.array([100e6, 300e6, 1e9])  # target scales: 100M / 300M / 1B knobs
labels = ['100M', '300M', '1B']  # short labels for the three targets
speed = rate / sizes  # projected tok/s on THIS cpu via the 1/params law
infer = 2 * sizes / 1e9  # inference GB: weights only, 2 bytes/knob (bf16)
train = 14 * sizes / 1e9  # training GB: weights + grads + Adam states, ~14 bytes/knob
print('proj tok/s:', [round(float(x), 1) for x in speed])  # print 285.5, 95.2, 28.5
print('infer GB:', infer.tolist())  # print 0.2, 0.6, 2.0: fits on one GPU
print('train GB:', train.tolist())  # print 1.4, 4.2, 14.0: needs a real machine
fig, ax = plt.subplots(1, 2, figsize=(10, 4))  # two panels: speed line + memory bars
ax[0].loglog([460416, 100e6, 300e6, 1e9], [62000, 285.5, 95.2, 28.5], 'o-')  # extrapolation from S
ax[0].set_xlabel('params (log scale)')  # label the size axis
ax[0].set_ylabel('tok/s (log scale)')  # label the speed axis
ax[0].set_title('Speed projection: 62000 -> 28.5 tok/s')  # title in printed numbers
ax[0].grid(True, alpha=0.3)  # add faint grid to read log values
x = np.arange(3)  # x slots for the three target scales
ax[1].bar(x - 0.2, infer, 0.4, label='infer (2 B/param)')  # short bars: just weights
ax[1].bar(x + 0.2, train, 0.4, label='train (14 B/param)')  # tall bars: weights + grads + Adam
ax[1].set_xticks(x, labels)  # name each scale under its pair
ax[1].set_ylabel('GB')  # label the memory axis
ax[1].set_title('Memory projection: 1B trains at 14.0 GB')  # title in printed numbers
ax[1].legend()  # show which bar is which budget
plt.tight_layout()  # snug the layout
plt.show()  # render the figure

## §8 How it all fits together — the system map

One lab run is a pipeline with every § term playing a station. **Config** picks depth/width (knob count); **train** streams TinyShakespeare characters (**tokens**, §4) through brief CPU loops and records **train loss**; **validate** re-scores on held-out text (**val loss**, gap alarm, §4); the **dashboard** rescales loss to **perplexity** (§5) and loss×speed into **tok/s scaling** (§6); **projection** stretches the anchor to 100M/300M/1B memory and speed (§7). The map below pins each term to its station; the numbers walk all three configs through it.


In [ ]:
import matplotlib.pyplot as plt  # only pyplot: this cell is a pure diagram
import random  # bring in stdlib random only to satisfy the seed habit (unused here)
random.seed(13)  # fix the seed for reproducibility
fig, ax = plt.subplots(figsize=(10, 3.5))  # wide figure for the left-to-right pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 14)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
stages = [('config\nXS/S/M', 1), ('train\ntokens §4', 3.5), ('loss\n§4 + §5', 6), ('dashboard\n§6 tok/s', 8.5), ('projection\n§7 1B', 11)]  # five pipeline stations
for name, xc in stages:  # draw each station box
    ax.text(xc, 2.2, name, ha='center', va='center', fontsize=8,  # centered station label
            bbox=dict(boxstyle='round', facecolor='lightblue'))  # blue = forward pipeline
for xa, xb in [(1.8, 2.7), (4.4, 5.2), (6.8, 7.7), (9.5, 10.2)]:  # arrow gaps between stations
    ax.annotate('', xy=(xb, 2.2), xytext=(xa, 2.2),  # draw a forward arrow left -> right
                arrowprops=dict(arrowstyle='->'))  # plain arrowhead style
ax.text(6, 3.4, 'FORWARD: bigger config, lower loss, slower tok/s (§4-§6)', ha='center', fontsize=9, fontweight='bold')  # phase banner
ax.text(6, 0.8, 'PROJECT: anchor S (2.85e10) -> 1B at 28.5 tok/s, 14.0 GB (§7)', ha='center', fontsize=9, fontweight='bold', color='darkred')  # projection banner
ax.set_title('System map: config -> train -> dashboard -> projection')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map

In [ ]:
import numpy as np  # reuse numpy for the config table math
np.random.seed(13)  # fix the seed for reproducibility
rate = 62000 * 460416  # reuse the S-anchored speed constant (§6)
configs = [('XS', 109824, 2.7944, 163000), ('S', 460416, 2.6177, 62000), ('M', 1204736, 2.6060, 30000)]  # (name, params, val, tok/s)
for name, p, v, t in configs:  # walk each config through every station
    ppl = round(float(np.exp(v)), 2)  # rescale loss to hesitation count (§5)
    mem = round(2 * p / 1e9, 4)  # inference GB at 2 bytes/knob (§7)
    print(name, 'val', v, '-> ppl', ppl, '| tok/s', t, '| infer GB', mem)  # one dashboard row per config
print('1B projection: tok/s', round(rate / 1e9, 1), '| train GB', 14.0)  # print the far extrapolation (§7)

## Bridge to the lab

Open `notebook.ipynb` now. Its cells implement exactly the map above, with real training instead of toy curves:

1. **Cell 1 (setup)** — seeds plus `os.makedirs('./results')`. Same `SEED = 13` habit as this lecture.
2. **Cell 2 (data)** — downloads TinyShakespeare (or a synthetic fallback) and splits 90/10 into train/val **tokens** (§4).
3. **Cell 3 (model factory)** — builds the XS/S/M students and verifies knob counts analytically; our anchor sizes are its exact numbers.
4. **Cell 4 (train each proxy)** — brief CPU training per config, recording **train/val loss** (§4), **perplexity** (§5), and **tok/s** (§6) into `results/configs.csv`.
5. **Cell 5 (dashboard)** — fits `tok/s × params = constant` (§6), projects 100M/300M/1B memory and speed (§7) into `results/projections.csv`, sweeps batch size, and saves `results/dashboard.png`.
6. **Cell 6 (cleanup)** — points at the CSVs/PNG and says: rerun on the DGX Spark and the tok/s rows jump 50–100x.


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **Train loss is 0.48 but val loss is 1.05. Did training fail?**

*Answer: no — training worked (guesses improved), but the 0.57 gap (§4) says the model memorized study text instead of generalizing. The best checkpoint was step 100 (val 0.99); past it, studying harder only memorizes harder. Trace it in the §8 numbers.*

2. **Val loss 2.61 — how confused is the model, in plain words?**

*Answer: perplexity is exp(2.606) ≈ 13.54 (§5) — at each step it hesitates between ~14 characters, down from ~61 when guessing randomly. Same information as the loss, in feelable units.*

3. **The S config does 62,000 tok/s. Why project only 28.5 tok/s for 1B, and what breaks first — memory or patience?**

*Answer: speed falls ~1/params (§6: constant 2.85e10 ÷ 1e9 ≈ 28.5), and training memory hits 14.0 GB (§7) before patience does — weights plus gradients plus Adam states at ~14 bytes/knob. That is why the lab projects instead of training.*
